In [ ]:
DATASET_VARIANT = "classification_model_originalimage"  # options: "classification_model", "classification_model_originalimage"


In [ ]:
#merge train and val dataset
import pandas as pd
import SimpleITK as sitk
import glob
import os

path = r"/path/to/workspace/classification_model_originalimage/"
train_paths = glob.glob(os.path.join(path, 'train_data', '*.nii.gz'))
# print(train_paths)
val_paths = glob.glob(os.path.join(path, 'val_data', '*.nii.gz'))
image_paths =train_paths + val_paths
print(image_paths)

In [ ]:
len(image_paths)

In [ ]:
import numpy as np
data_summary = pd.DataFrame(columns=["Image", "Spacing", "Dimension", "Mean", "Std", "Min", "Max"])

data_list = []

for img_path in image_paths:
    img = sitk.ReadImage(img_path)
    spacing = img.GetSpacing()
    dimension = img.GetSize()
    img_data = sitk.GetArrayFromImage(img)
    data_list.append({
        "Image": os.path.basename(img_path),
        "Spacing": spacing,
        "Dimension": dimension,
        "Mean": np.mean(img_data),
        "Std": np.std(img_data),
        "Min": np.min(img_data),
        "Max": np.max(img_data)
    })

print("Done")

In [ ]:
#save csv to path
data_summary = pd.DataFrame(data_list)
output_dir = r"/path/to/workspace/classification_model_originalimage"
output_path = os.path.join(output_dir, "dataset_summary_train_val2.csv")
data_summary.to_csv(output_path, index=False)

In [ ]:
# analyze percentile

# data_summary = pd.read_csv("data_summary_train.csv")
# data_summary = pd.read_csv("data_summary_train_val.csv")
data_summary['Spacing'] = data_summary['Spacing'].astype(str)
data_summary[['Spacing_x', 'Spacing_y', 'Spacing_z']] = data_summary['Spacing'].str.strip("()").str.split(",", expand=True).astype(float)
spacing_a = data_summary[['Spacing_x', 'Spacing_y', 'Spacing_z']].quantile([0.25, 0.5, 0.75])

data_summary['Dimension'] = data_summary['Dimension'].astype(str)
data_summary[['Dimension_x', 'Dimension_y', 'Dimension_z']] = data_summary['Dimension'].str.strip("()").str.split(",", expand=True).astype(float)
dimension_a = data_summary[['Dimension_x', 'Dimension_y', 'Dimension_z']].quantile([0.25, 0.5, 0.75])

dimension_max = data_summary[['Dimension_x', 'Dimension_y', 'Dimension_z']].max()
dimension_min = data_summary[['Dimension_x', 'Dimension_y', 'Dimension_z']].min()

print("spacing_a:\n", spacing_a)
print()
print("dimension_a:\n", dimension_a)
print()
print("Dimension Max:\n", dimension_max)
print()
print("Dimension Min:\n", dimension_min)
print()

mean_a = data_summary['Mean'].describe(percentiles=[0.25, 0.5, 0.75])
std_a = data_summary['Std'].describe(percentiles=[0.25, 0.5, 0.75])
min_a = data_summary['Min'].describe(percentiles=[0.25, 0.5, 0.75])
max_a = data_summary['Max'].describe(percentiles=[0.25, 0.5, 0.75])

print("mean_a:\n", mean_a)
print()
print("std_a:\n", std_a)
print()
print("min_a:\n", min_a)
print()
print("max_a:\n", max_a)
print()